# Group 13 — Phase 3 (Step 3): Full Metric Panel, Clean T4 Latency & Patch-Occlusion Fidelity

**Repository:** `https://github.com/Uyhuynh23/Group13_CV` (branch `main`)  
**Runtime:** Google Colab **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`)  
**Drive Data:** `/content/drive/MyDrive/Group13_CV_Data`

## Step 1: Mount Google Drive & Verify T4 GPU

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CUDA available: True
Device: Tesla T4


## Step 2: Clone/Pull GitHub Repo (`main`) & Install Dependencies

In [3]:
import os
if not os.path.exists('/content/Group13_CV'):
    !git clone -b main https://github.com/Uyhuynh23/Group13_CV.git /content/Group13_CV
%cd /content/Group13_CV
!git pull origin main
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/openai/CLIP.git

Cloning into '/content/Group13_CV'...
remote: Enumerating objects: 241, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 241 (delta 114), reused 129 (delta 90), pack-reused 76 (from 1)
Receiving objects: 100% (241/241), 2.38 MiB | 23.18 MiB/s, done.
Resolving deltas: 100% (119/119), done.
/content/Group13_CV
From https://github.com/Uyhuynh23/Group13_CV
 * branch            main       -> FETCH_HEAD
Already up to date.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Step 3: Clean T4 Latency Re-benchmark (`baseline` & `condD` seed 123)

In [4]:
DRIVE_DATA = '/content/drive/MyDrive/Group13_CV_Data'
CACHE_PATH = f'{DRIVE_DATA}/cache/test_cache.pt'
CKPT_DIR = f'{DRIVE_DATA}/checkpoints'
PHASE3_OUT = f'{DRIVE_DATA}/phase3_results'
!mkdir -p {PHASE3_OUT}

!python evaluate.py --checkpoint {CKPT_DIR}/baseline_global_linear_seed42_best.pt --cache_path {CACHE_PATH} --out_dir {PHASE3_OUT} --beam_size 5
!python evaluate.py --checkpoint {CKPT_DIR}/condD_query_resampler_lora_seed123_best.pt --cache_path {CACHE_PATH} --out_dir {PHASE3_OUT} --beam_size 5
!python evaluate.py --checkpoint {CKPT_DIR}/condD_query_resampler_lora_seed123_best.pt --cache_path {CACHE_PATH} --out_dir {PHASE3_OUT} --beam_size 1

Loading checkpoint from /content/drive/MyDrive/Group13_CV_Data/checkpoints/baseline_global_linear_seed42_best.pt on cuda ...
config.json: 100% 665/665 [00:00<00:00, 2.15MB/s]

model.safetensors: downloading bytes:  42% 230M/548M [00:00<00:00, 422MB/s, 16.3MB/s  ]
model.safetensors: reconstructing file:  24% 134M/548M [00:00<00:02, 142MB/s]
model.safetensors: downloading bytes:  83% 454M/548M [00:01<00:00, 455MB/s, 37.6MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:01<00:00, 257MB/s, 44.1MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:01<00:00, 297MB/s, 51.6MB/s  ]
Loading weights: 100% 148/148 [00:00<00:00, 4959.95it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 588kB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 112kB/s]
vocab.json: 100% 1.04M/1.04M [00:00<00:00, 22.0MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 14.4MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 30.0MB/s]
Evaluating test (beam=5): 100% 1000/1000 [03:58<00:00

## Step 4: Full Metric Panel (METEOR, SPICE, CLIPScore `ViT-B/16`)
Note: If raw Flickr8k images are in `/content/drive/MyDrive/Group13_CV_Data/flickr8k_zips`, unzip them to `/content/flickr8k_raw` first for `--compute_clipscore`.

In [5]:
import glob, os

RAW_DATA_DIR = '/content/flickr8k_raw'
if not os.path.exists(f'{RAW_DATA_DIR}/Flicker8k_Dataset'):
    !mkdir -p {RAW_DATA_DIR}
    !unzip -q -o {DRIVE_DATA}/flickr8k_zips/Flickr8k_Dataset.zip -d {RAW_DATA_DIR}

ckpts = sorted(glob.glob(f'{CKPT_DIR}/*_best.pt'))
print(f'Found {len(ckpts)} best checkpoints.')
for ckpt in ckpts:
    print('Evaluating:', os.path.basename(ckpt))
    !python evaluate.py --checkpoint {ckpt} --cache_path {CACHE_PATH} --data_dir {RAW_DATA_DIR} --out_dir {PHASE3_OUT}/full_metrics --beam_size 5 --compute_spice --compute_clipscore

Found 13 best checkpoints.
Evaluating: baseline_global_linear_seed42_best.pt
Loading checkpoint from /content/drive/MyDrive/Group13_CV_Data/checkpoints/baseline_global_linear_seed42_best.pt on cuda ...
Loading weights: 100% 148/148 [00:00<00:00, 4800.83it/s]
Evaluating test (beam=5): 100% 1000/1000 [03:40<00:00,  4.53it/s]
PTBTokenizer tokenized 64178 tokens at 420213.76 tokens per second.
PTBTokenizer tokenized 13231 tokens at 138981.19 tokens per second.
{'testlen': 11136, 'reflen': 10203, 'guess': [11136, 10136, 9136, 8136], 'correct': [4530, 899, 139, 56]}
ratio: 1.0914436930313542
[WARN] Java METEOR failed (could not convert string to float: b'6.0 12.0 3.0 5.0 1.0 1.0 1.0 1.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 2.0 2.0 2.0'); falling back to NLTK METEOR.
[WARN] SPICE computation skipped: <urlopen error [Errno 113] No route to host>
Computing reference-free CLIPScore using ViT-B/16 ...
100%|████████████████████████████████████████| 335M/335M [00:01<00:00, 201MiB/s]
/usr

## Step 5: Patch-Occlusion Fidelity ($\Delta \log p(w)$) on 100-Image Audit Subset

In [10]:
import json, random, torch
import numpy as np
from transformers import GPT2Tokenizer
from models import CaptionBaselineModel

def load_model(ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    ckpt_args = ckpt.get('args', {})
    model = CaptionBaselineModel(
        connector_type=ckpt_args.get('connector_type', 'global_linear'),
        prefix_length=ckpt_args.get('prefix_length', 1),
        use_lora=ckpt_args.get('use_lora', False),
        lora_r=ckpt_args.get('lora_r', 8),
        lora_alpha=ckpt_args.get('lora_alpha', 16),
        lora_dropout=ckpt_args.get('lora_dropout', 0.05),
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device)
    model.eval()
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_cache = torch.load(CACHE_PATH, map_location='cpu')
with open('/content/Group13_CV/phase3_outputs/rome_audit_blinded.json') as f:
    audit_items = json.load(f)
audit_ids = [x['image_id'] for x in audit_items]

id_to_idx = {img_id: i for i, img_id in enumerate(test_cache['image_ids'])}
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

def run_occlusion(ckpt_path, k_mask=10, seed=42):
    rng = random.Random(seed)
    model = load_model(ckpt_path, device)
    drops = {'top': [], 'low': [], 'random': []}
    for img_id in audit_ids:
        idx = id_to_idx[img_id]
        patches = test_cache['patch_features'][idx].unsqueeze(0).to(device, dtype=torch.float32)
        with torch.no_grad():
            # Baseline log-prob of first generated content token
            prefix = model.connector(patches)
            out = model.gpt(inputs_embeds=prefix, output_attentions=True)
            next_logits = out.logits[:, -1, :]
            target_tok = next_logits.argmax(dim=-1)
            base_lp = torch.log_softmax(next_logits, dim=-1)[0, target_tok[0]].item()
            # Patch importance via L2 norm of patch-to-prefix gradient/activation proxy
            scores = patches.norm(dim=-1).squeeze(0).cpu().numpy()
            ranked = np.argsort(scores)[::-1].tolist()
            top_idx = ranked[:k_mask]
            low_idx = ranked[-k_mask:]
            rand_idx = rng.sample(range(49), k_mask)
            for mode, mask_indices in [('top', top_idx), ('low', low_idx), ('random', rand_idx)]:
                masked = patches.clone()
                masked[:, mask_indices, :] = 0.0
                m_out = model.gpt(inputs_embeds=model.connector(masked))
                m_lp = torch.log_softmax(m_out.logits[:, -1, :], dim=-1)[0, target_tok[0]].item()
                drops[mode].append(base_lp - m_lp)
    return {m: {'mean_drop': float(np.mean(v)), 'std': float(np.std(v))} for m, v in drops.items()}

occ_results = {}
for name in ['condA_patch_mlp_frozen_seed42_best.pt', 'condB_query_resampler_frozen_seed42_best.pt', 'condD_query_resampler_lora_seed42_best.pt']:
    occ_results[name] = run_occlusion(f'{CKPT_DIR}/{name}')
    print(name, occ_results[name])

with open(f'{PHASE3_OUT}/occlusion_fidelity.json', 'w') as f:
    json.dump(occ_results, f, indent=2)
print('Saved occlusion_fidelity.json!')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

condA_patch_mlp_frozen_seed42_best.pt {'top': {'mean_drop': 5.918496548868716, 'std': 5.724117594927756}, 'low': {'mean_drop': 0.5189277613908052, 'std': 1.9416565912901}, 'random': {'mean_drop': 2.1508026936650277, 'std': 4.0934232999553215}}


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

condB_query_resampler_frozen_seed42_best.pt {'top': {'mean_drop': 0.05510239385068417, 'std': 0.24428875201602568}, 'low': {'mean_drop': 0.036482049077749254, 'std': 0.31049606921580614}, 'random': {'mean_drop': 0.01300482526421547, 'std': 0.09459809659009381}}


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

condD_query_resampler_lora_seed42_best.pt {'top': {'mean_drop': 0.05261638730764389, 'std': 0.21053930060419798}, 'low': {'mean_drop': 0.04282351821660996, 'std': 0.3578784202834899}, 'random': {'mean_drop': 0.022641153633594514, 'std': 0.09002646884662854}}
Saved occlusion_fidelity.json!
